In [ ]:
"""
Created on Sun May 24 18:25:50 2026

@author: PC
"""
#__init__.py
# -*- coding: utf-8 -*-
"""
Created on Sun May 24 18:25:50 2026

@author: PC
"""


#data.py
# -*- coding: utf-8 -*-

import torch
import torchvision
import torchvision.transforms as transforms
# -----------------------------------
# Transformations
# -----------------------------------

normalTransformation = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

augmentationTransformation = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# -----------------------------------
# Load Data
# -----------------------------------

def get_dataloaders(useAugmentation=False, batchSize=128):

    transformation = (
        augmentationTransformation
        if useAugmentation
        else normalTransformation
    )

    trainDataset = torchvision.datasets.FashionMNIST(
        root='./data',
        train=True,
        download=True,
        transform=transformation
    )

    trainLoader = torch.utils.data.DataLoader(
        dataset=trainDataset,
        batch_size=batchSize,
        shuffle=True
    )

    testDataset = torchvision.datasets.FashionMNIST(
        root='./data',
        train=False,
        download=True,
        transform=normalTransformation
    )

    testLoader = torch.utils.data.DataLoader(
        dataset=testDataset,
        batch_size=batchSize,
        shuffle=False
    )

    return trainLoader, testLoader

# -----------------------------------
# Classes
# -----------------------------------

classes = (
    'T-shirt/top',
    'Trouser',
    'Pullover',
    'Dress',
    'Coat',
    'Sandal',
    'Shirt',
    'Sneaker',
    'Bag',
    'Ankle boot'
)

In [ ]:
#model.py
# -*- coding: utf-8 -*-

import torch
import torch.nn as nn
import torch.nn.functional as F

class Model(nn.Module):

    def __init__(self):
        super(Model, self).__init__()

        # First Block
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 32, kernel_size=3)
        self.bn2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(
            32,
            32,
            kernel_size=5,
            stride=2,
            padding=2
        )

        self.bn3 = nn.BatchNorm2d(32)

        self.drop1 = nn.Dropout(0.4)

        # Second Block
        self.conv4 = nn.Conv2d(32, 64, kernel_size=3)
        self.bn4 = nn.BatchNorm2d(64)

        self.conv5 = nn.Conv2d(64, 64, kernel_size=3)
        self.bn5 = nn.BatchNorm2d(64)

        self.conv6 = nn.Conv2d(
            64,
            64,
            kernel_size=5,
            stride=2,
            padding=2
        )

        self.bn6 = nn.BatchNorm2d(64)

        self.drop2 = nn.Dropout(0.4)

        # Third Block
        self.conv7 = nn.Conv2d(64, 128, kernel_size=4)
        self.bn7 = nn.BatchNorm2d(128)

        self.drop3 = nn.Dropout(0.4)

        # Fully Connected Layer
        self.fc = nn.Linear(128, 10)

    def forward(self, x):

        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.drop1(x)

        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        x = F.relu(self.bn6(self.conv6(x)))
        x = self.drop2(x)

        x = F.relu(self.bn7(self.conv7(x)))

        x = torch.flatten(x, 1)

        x = self.drop3(x)

        x = self.fc(x)

        return x


In [ ]:
#evaluate.py


#from src.data import classes

# -----------------------------------
# ACCURACY
# -----------------------------------

def evaluate_accuracy(model, loader, device):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total


# -----------------------------------
# LOSS
# -----------------------------------

def evaluate_loss(model, loader, criterion, device):

    model.eval()

    total_loss = 0.0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item()

    return total_loss / len(loader)


# -----------------------------------
# CONFUSION MATRIX
# -----------------------------------

def confusion_matrix(model, loader, device):

    num_classes = len(classes)

    cm = torch.zeros((num_classes, num_classes), dtype=torch.int64)

    model.eval()

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, preds = torch.max(outputs, 1)

            for t, p in zip(labels, preds):
                cm[t, p] += 1

    return cm


# -----------------------------------
# CLASS ACCURACY
# -----------------------------------

def class_accuracy(confusionMatrix):

    classAccuracies = {}

    for i, className in enumerate(classes):

        correct = confusionMatrix[i][i].item()
        total = confusionMatrix[i].sum().item()

        classAccuracies[className] = (
            100 * correct / total if total > 0 else 0
        )

    return classAccuracies

In [ ]:
#train.py
# -*- coding: utf-8 -*-

# from src.data import get_dataloaders
# from src.model import Model
# from src.evaluate import evaluate_accuracy, evaluate_loss

import torch
import torch.optim as optim
import torch.nn as nn
import os
# Create folder for saving models
os.makedirs("models", exist_ok=True)
def train_model(
    useAugmentation=False,
    batchSize=128,
    epochs=10
):

    print("Starting training process...\n")

    criterion = nn.CrossEntropyLoss()

    device = torch.device(
        'cuda:0'
        if torch.cuda.is_available()
        else 'cpu'
    )

    print(f"Using device: {device}\n")

    net = Model().to(device)

    trainLoader, testLoader = get_dataloaders(
        useAugmentation,
        batchSize
    )

    optimizer = optim.Adam(
        net.parameters(),
        lr=0.001,
        weight_decay=0.0001
    )

    scheduler = optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda epoch: 0.95 ** epoch
    )

    lossHistory = []
    testLossHistory = []
    accuracyHistory = []

    bestAccuracy = 0.0
    bestModelState = None

    # -----------------------------------
    # TRAINING LOOP
    # -----------------------------------

    for epoch in range(epochs):

        net.train()

        runningLoss = 0.0

        for inputs, labels in trainLoader:

            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = net(inputs)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            runningLoss += loss.item()

        scheduler.step()

        # Average training loss
        trainLoss = runningLoss / len(trainLoader)

        # Evaluate test loss
        testLoss = evaluate_loss(
            net,
            testLoader,
            criterion,
            device
        )

        # Evaluate accuracy
        testAccuracy = evaluate_accuracy(
            net,
            testLoader,
            device
        )

        # Save best model
        if testAccuracy > bestAccuracy:
            bestAccuracy = testAccuracy
            bestModelState = net.state_dict()
            savePath = os.path.join("models", "best_fashionmnist_model.pth")
            torch.save(net.state_dict(), savePath)
            print(f"Best model saved at: {savePath}")

        # Save histories
        lossHistory.append(trainLoss)
        testLossHistory.append(testLoss)
        accuracyHistory.append(testAccuracy)

        # Print epoch results
        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {trainLoss:.4f} | "
            f"Test Loss: {testLoss:.4f} | "
            f"Test Accuracy: {testAccuracy:.2f}%"
        )

    # Load best model
    net.load_state_dict(bestModelState)

    print("\nTraining finished.")
    print(f"Best Test Accuracy: {bestAccuracy:.2f}%")

    return (
        net,
        lossHistory,
        testLossHistory,
        accuracyHistory,
        device,
        testLoader
    )


In [ ]:
#utils.py

# -*- coding: utf-8 -*-

import matplotlib.pyplot as plt
import random
#import numpy as np


# -----------------------------------
# CONFUSION MATRIX
# -----------------------------------

def show_confusionMatrix(cm, class_names=None):

    cm = cm.cpu().numpy()

    if class_names is None:
        class_names = [str(i) for i in range(len(cm))]

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, cmap="Blues")

    plt.title("Confusion Matrix")
    plt.colorbar()

    plt.xticks(range(len(class_names)), class_names, rotation=45)
    plt.yticks(range(len(class_names)), class_names)

    # values inside cells
    for i in range(len(cm)):
        for j in range(len(cm)):
            plt.text(
                j, i,
                cm[i, j],
                ha="center",
                va="center",
                color="black"
            )

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()


# -----------------------------------
# LOSS + ACCURACY PLOT (FIXED PART)
# -----------------------------------

def plot_metrics(trainLoss, testLoss, testAcc):

    epochs = range(1, len(trainLoss) + 1)

    plt.figure(figsize=(14, 5))

    # -----------------------
    # LOSS PLOT
    # -----------------------
    plt.subplot(1, 2, 1)
    plt.plot(epochs, trainLoss, label="Train Loss")
    plt.plot(epochs, testLoss, label="Test Loss")

    plt.title("Loss Curve")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)

    # -----------------------
    # ACCURACY PLOT
    # -----------------------
    plt.subplot(1, 2, 2)
    plt.plot(epochs, testAcc, label="Test Accuracy", color="green")

    plt.title("Test Accuracy Curve")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy (%)")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()
    
def show_random_predictions(model, loader, device, class_names, num_images=5):
    model.eval()

    images_list = []
    labels_list = []
    preds_list = []

    # collect one batch (or multiple batches if needed)
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            images_list.append(images.cpu())
            labels_list.append(labels.cpu())
            preds_list.append(preds.cpu())

            if len(torch.cat(images_list)) >= num_images:
                break

    # flatten collected tensors
    images = torch.cat(images_list)[:num_images]
    labels = torch.cat(labels_list)[:num_images]
    preds = torch.cat(preds_list)[:num_images]

    # plot
    plt.figure(figsize=(12, 3))

    for i in range(num_images):
        img = images[i].squeeze()

        plt.subplot(1, num_images, i + 1)
        plt.imshow(img, cmap="gray")

        plt.title(
            f"T:{class_names[labels[i]]}\nP:{class_names[preds[i]]}",
            fontsize=9
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
#main.py
# -*- coding: utf-8 -*-

# from src.train import train_model
# from src.evaluate import confusion_matrix, class_accuracy
# from src.utils import plot_metrics, show_confusionMatrix
# -*- coding: utf-8 -*-

def main():

    (
        model,
        lossHistory,
        testLossHistory,
        accuracyHistory,
        device,
        testLoader
    ) = train_model(
        useAugmentation=False,
        batchSize=128,
        epochs=10
    )

    # -----------------------------------
    # LOSS + ACCURACY PLOTS
    # -----------------------------------

    plot_metrics(
        lossHistory,
        testLossHistory,
        accuracyHistory
    )

    # -----------------------------------
    # CONFUSION MATRIX
    # -----------------------------------

    cm = confusion_matrix(
        model,
        testLoader,
        device
    )

    print("\nConfusion Matrix:\n")
    print(cm.cpu().numpy())

    show_confusionMatrix(cm, class_names=classes)

    # -----------------------------------
    # CLASS ACCURACY
    # -----------------------------------

    classAccuracies = class_accuracy(cm)

    print("\nPer-Class Accuracy:\n")

    for className, accuracy in classAccuracies.items():
        print(f"{className}: {accuracy:.2f}%")
        
    show_random_predictions(
    model,
    testLoader,
    device,
    classes,
    num_images=5)


if __name__ == "__main__":
    main()